# 04 — Calendar effects (H4)

> **Data notice.** Values are real daily revenue multiplied by a single constant scaling factor for privacy. All statistical properties (patterns, breaks, tests, coefficients) are identical to those on the raw data — only the absolute currency level is hidden. Units labeled **LC** (local currency, scaled). Weekday labels: **BD1–BD5** for business days, **WE1–WE2** for weekend days, in business-week order.

**Hypotheses:**

**H4a (coarse):** revenue on weekend days is significantly lower than on business days.

**H4a (refinement, data-driven):** the two weekend days are NOT interchangeable — one shows a much sharper drop than the other.

**H4b:** two competing priors on the payday-window question:

- *Textbook prior:* revenue in the payday window (last 5 + first 5 days of each month) is ≥10% higher than the middle of the month, driven by a monthly salary-cycle discretionary purchase pattern.
- *Institutional prior:* a public prescription-reimbursement reform, introduced several years before this sample, replaced full out-of-pocket prescription costs with a small copay (0-20%). The mechanism decouples prescription-purchase timing from income timing. Under this prior, the payday cycle is predicted to be absent.
We test which prior the data supports.

**Method.** OLS regression with dummy variables and **HAC (Newey-West) standard errors** (lag = 7). HAC is required because daily revenue is autocorrelated and classical OLS SEs would understate coefficient variance, inflating t-statistics.

**Controls:** all models include a `covid_flag` dummy (post-2020-03-26) to absorb the level shift established in notebook 02 — so calendar coefficients are estimated net of the COVID effect.

**EDA expectations:** H4a coarse TRUE with an **asymmetric** refinement — the EDA day-of-week chart already visually suggests the two weekend days differ; the formal regression quantifies it. H4b **likely FALSE** — no month-phase cycle was visible in EDA.

**Approach:** data prep with flag columns → H4a coarse (single weekend dummy) → H4a refinement (separate WE1/WE2 dummies) → H4b (payday window) → joint model → verdict.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['figure.dpi'] = 110

## Load and prepare data

We load the same scaled daily series used throughout the project and build the flag columns needed for the regressions:

| Column | Meaning |
|---|---|
| `dow_label` | `BD1`–`BD5` or `WE1`–`WE2`, in business-week order |
| `is_weekend` | 1 if weekend day, 0 otherwise (used in H4a coarse) |
| `we1_flag` | 1 if first weekend day (WE1), 0 otherwise (used in H4a refinement) |
| `we2_flag` | 1 if second weekend day (WE2), 0 otherwise (used in H4a refinement) |
| `day_of_month` | 1–31 |
| `payday_window` | 1 if day is in first 5 OR last 5 days of the month, 0 otherwise (H4b) |
| `covid_flag` | 1 if date ≥ 2020-03-26, 0 otherwise (control) |

In [2]:
df = pd.read_csv('../data/daily_sales_final.csv', parse_dates=['date'])
df = df.set_index('date').sort_index()
DOW_MAP = {
    6: 'BD1',
    0: 'BD2',
    1: 'BD3',
    2: 'BD4',
    3: 'BD5',
    4: 'WE1',
    5: 'WE2',
}
df['dow_label'] = df.index.dayofweek.map(DOW_MAP)

df['is_weekend'] = df['dow_label'].isin(['WE1', 'WE2']).astype(int)
df['we1_flag']   = (df['dow_label'] == 'WE1').astype(int)
df['we2_flag']   = (df['dow_label'] == 'WE2').astype(int)

df['day_of_month'] = df.index.day
df['days_in_month'] = df.index.days_in_month
df['payday_window'] = (
    (df['day_of_month'] <= 5) |
    (df['day_of_month'] > df['days_in_month'] - 5)
).astype(int)

df['covid_flag'] = (df.index >= '2020-03-26').astype(int)

y = df['revenue_scaled']
print(df[['dow_label', 'is_weekend', 'we1_flag', 'we2_flag',
          'day_of_month', 'payday_window', 'covid_flag']].head(15))

           dow_label  is_weekend  we1_flag  we2_flag  day_of_month  \
date                                                                 
2019-12-15       BD1           0         0         0            15   
2019-12-16       BD2           0         0         0            16   
2019-12-17       BD3           0         0         0            17   
2019-12-18       BD4           0         0         0            18   
2019-12-19       BD5           0         0         0            19   
2019-12-20       WE1           1         1         0            20   
2019-12-21       WE2           1         0         1            21   
2019-12-22       BD1           0         0         0            22   
2019-12-23       BD2           0         0         0            23   
2019-12-24       BD3           0         0         0            24   
2019-12-25       BD4           0         0         0            25   
2019-12-26       BD5           0         0         0            26   
2019-12-27       WE1

### Sanity check on the flag columns

In [3]:
print('Counts by day-of-week label:')
print(df['dow_label'].value_counts().reindex(['BD1','BD2','BD3','BD4','BD5','WE1','WE2']))

print(f"\nWeekend days: {df['is_weekend'].sum()} / {len(df)} "
      f"({100*df['is_weekend'].mean():.1f}%)")
print(f"Payday-window days: {df['payday_window'].sum()} / {len(df)} "
      f"({100*df['payday_window'].mean():.1f}%)")
print(f"Post-COVID days: {df['covid_flag'].sum()} / {len(df)} "
      f"({100*df['covid_flag'].mean():.1f}%)")

Counts by day-of-week label:
dow_label
BD1    68
BD2    68
BD3    66
BD4    67
BD5    67
WE1    67
WE2    67
Name: count, dtype: int64

Weekend days: 134 / 470 (28.5%)
Payday-window days: 150 / 470 (31.9%)
Post-COVID days: 375 / 470 (79.8%)


## H4a coarse — weekend vs business day

In [4]:
X = sm.add_constant(df[['is_weekend', 'covid_flag']])
res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 7})
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:         revenue_scaled   R-squared:                       0.261
Model:                            OLS   Adj. R-squared:                  0.257
Method:                 Least Squares   F-statistic:                     53.52
Date:                Tue, 15 Sep 2026   Prob (F-statistic):           1.18e-21
Time:                        03:07:31   Log-Likelihood:                -5374.7
No. Observations:                 470   AIC:                         1.076e+04
Df Residuals:                     467   BIC:                         1.077e+04
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.472e+05   3463.099     42.518      0.0

The OLS regression shows that weekend average daily revenue is below that of business days by around 13,120 LC — a 8.9% drop. The effect is significant at p < 0.01 with a wide margin (z = -8.14): H4a coarse is confirmed. Durbin-Watson at 1.234 indicates moderately strong positive autocorrelation in the model's residuals (a value of 2 corresponds to no autocorrelation) — without HAC, the classical OLS SEs on these coefficients would have been badly understated. Jarque-Bera = 163 with p ≈ 3e-36 shows the residuals are strongly non-normal (skew 0.60, kurtosis 5.63); this means classical prediction intervals for individual observations would be wrong, but it does not invalidate the coefficient estimates or the HAC SEs, since OLS with HAC does not require normality for asymptotic inference. The next section addresses whether the two days of the weekend contribute equally to this drop.

## H4a refinement — weekend asymmetry

In [5]:
X = sm.add_constant(df[['we1_flag', 'we2_flag', 'covid_flag']])
res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 7})
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:         revenue_scaled   R-squared:                       0.353
Model:                            OLS   Adj. R-squared:                  0.348
Method:                 Least Squares   F-statistic:                     110.0
Date:                Tue, 15 Sep 2026   Prob (F-statistic):           7.30e-54
Time:                        03:07:31   Log-Likelihood:                -5343.4
No. Observations:                 470   AIC:                         1.069e+04
Df Residuals:                     466   BIC:                         1.071e+04
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.472e+05   3465.411     42.489      0.0

In [6]:
wald = res.wald_test('we1_flag = we2_flag', scalar=True)
print(f'\nWald test (we1_flag = we2_flag): F = {wald.statistic:.3f}, p = {wald.pvalue:.4f}')


Wald test (we1_flag = we2_flag): F = 205.717, p = 0.0000


The OLS results show that the weekend's first day (WE1) average revenue is 27,910 LC below the business-day baseline — a 19% drop — with p < 0.01 and z = -15.11. In contrast, the weekend's second day (WE2) sits within noise of the business-day baseline (+1,680 LC, ~+1%) and is statistically indistinguishable from a business day (p = 0.37). The entire weekend depression is concentrated on the first day. The Wald test of equality between WE1 and WE2 gives F = 205.7, p ≈ 0 — the two weekend days are formally, massively different, and the asymmetry cannot be explained by noise. R² rises from 0.261 to 0.353, quantifying the additional structure captured by splitting the two weekend days into separate regressors. The operator's prior (weekend drop) was correct at the coarse level. The EDA day-of-week chart visually suggested the two weekend days differed; the regression here quantifies and confirms the asymmetry rigorously — the depression is entirely concentrated on WE1 (-19%, highly significant), while WE2 sits statistically at the business-day level.



## H4b — payday effect

Under the textbook prior, we expect a positive `payday_window` coefficient of ≥10% of the baseline, significant at p < 0.01. Under the institutional prior (see hypothesis block above), we expect the coefficient near zero and not statistically distinguishable from zero.


In [7]:
X = sm.add_constant(df[['payday_window', 'we1_flag', 'we2_flag', 'covid_flag']])
res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 7})
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:         revenue_scaled   R-squared:                       0.354
Model:                            OLS   Adj. R-squared:                  0.348
Method:                 Least Squares   F-statistic:                     82.86
Date:                Tue, 15 Sep 2026   Prob (F-statistic):           4.52e-53
Time:                        03:07:31   Log-Likelihood:                -5342.9
No. Observations:                 470   AIC:                         1.070e+04
Df Residuals:                     465   BIC:                         1.072e+04
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const          1.477e+05   3659.981     40.368

The table shows that average daily revenue during the payday window is 2,080 LC (-1.4%) below the baseline, with z = -0.617 and p = 0.537. The point estimate is not only insignificant — its sign is negative, the opposite of the predicted positive payday boost. Even setting aside statistical significance, the data points against H4b, not merely fails to support it. R² moves negligibly (0.353 → 0.354), meaning the payday_window regressor adds 0.1 percentage point of explained variance — effectively zero. The WE1, WE2, and COVID coefficients are basically unchanged from the H4a refinement, confirming that this null is a genuine "does not matter" result and not an artifact of controls fighting each other.

The absence of a payday cycle has a specific institutional explanation the operator can attest to. Historically — before the rollout of a public health-insurance reform several years prior to the sample window — prescription purchases were paid in full out-of-pocket, and the operator observed a distinct payday cycle in his revenue. The reform shifted prescription costs to a small copay (typically 0-20% of retail), decoupling prescription-purchase timing from income timing. After the transition, the payday cycle disappeared from the revenue series. The dataset here starts well after the transition — every day in the sample sits in the post-reform regime, so the observed null is exactly what this mechanism predicts. Two secondary factors likely reinforce the effect: (i) chronic-prescription refills dominate the revenue mix and are driven by medical cycles rather than payroll, and (ii) local income patterns may not sync tightly to a monthly salary calendar.

The observed result — coefficient -1.4%, sign wrong, p = 0.537 — is inconsistent with the textbook prior (which required a positive coefficient ≥10% and p < 0.01) and consistent with the institutional prior. The reform mechanism dominates. H4b in its textbook form is FALSE; the institutional prior is confirmed.


## Verdict

**H4a coarse — CONFIRMED.** Weekend days average 13,120 LC (-8.9%) below business days, HAC z = -8.14, p < 0.01. The operator's coarse prior holds.

**H4a refinement — CONFIRMED as asymmetric.** The weekend depression is entirely concentrated on WE1 (-27,910 LC, -19%, z = -15.11, p < 0.01). WE2 is statistically indistinguishable from a business day (+1,680 LC, +1.1%, p = 0.37). The Wald test on WE1 = WE2 rejects equality with F = 205.7, p ≈ 0. R² rises from 0.261 to 0.353 when the refinement is added — 9 percentage points of additional explained variance. The operator's coarse prior (weekend depression) held; the EDA day-of-week chart visually suggested the two weekend days differed; the regression here quantifies the asymmetry rigorously and confirms its statistical significance.

**H4b — FALSE in its textbook form; institutional prior CONFIRMED.** The `payday_window` coefficient is -2,080 LC (-1.4%, p = 0.537) — sign wrong, magnitude negligible, statistically insignificant. R² barely moves. The observed null is consistent with an institutional mechanism the operator can attest to: a public health-insurance reform, introduced years before the sample, shifted prescription payments from full price out-of-pocket to a small copay, decoupling prescription-purchase timing from income timing. The observed null is exactly what this mechanism predicts. Two competing priors were tested; the institutional one wins.

**Methodological note.** Every model in this notebook was fitted with HAC (Newey-West) standard errors, lag = 7. Durbin-Watson statistics of 1.02–1.23 confirm strong positive residual autocorrelation — under classical OLS SEs, the coefficient variances would have been badly understated and every t-statistic reported here would have been inflated. HAC is the correct professional tool for daily-frequency time series; using classical OLS SEs would have been a substantive methodological error.
